# 📘 Lesson 02 — Permutation Importance
- Explainability Machine Learning — Study Companion (Moacir)
- Hybrid Learning Notebook — Study + Reference + Hands-on

## 📚 Curso Explainability Machine Learning

Este notebook segue o mesmo estilo pedagógico das lessons anteriores:

> **Permutation Importance — entendendo quais features o modelo realmente usa**

Ele funciona como:
- capítulo de estudo (explicações conceituais)
- guia de referência rápida (API-style)
- laboratório prático (código executável)

E está totalmente alinhado ao dataset sintético:
**medical_readmissions.csv**

# 1) Objetivos da Lesson

- Entender o conceito de *feature importance*.
- Compreender o funcionamento da técnica **Permutation Importance**.
- Calcular permutation importance em um modelo treinado.
- Interpretar corretamente os valores de importância.
- Identificar features relevantes, irrelevantes e redundantes.
- Preparar terreno para técnicas mais avançadas (PDP, SHAP).

# 2) Glossário Técnico

- **Feature Importance** — medida de quanto uma feature contribui para o modelo.
- **Permutation Importance** — técnica que mede a queda de performance ao embaralhar uma feature.
- **Baseline** — performance original do modelo antes do embaralhamento.
- **Métrica (Score)** — função usada para avaliar o modelo (ex.: accuracy).
- **Validação** — conjunto de dados usado para avaliar o modelo após o treino.
- **Embaralhamento (Permutation)** — reordenação aleatória dos valores de uma coluna.

# 3) Mini‑Referência (API Style)

- `pandas.read_csv(path)`
- `train_test_split(X, y, random_state=...)`
- `RandomForestClassifier(n_estimators=..., random_state=...)`
- `RandomForestClassifier.fit(X, y)`
- `RandomForestClassifier.predict(X)`
- `accuracy_score(y_true, y_pred)`
- `eli5.sklearn.PermutationImportance(estimator, random_state=...)`
- `eli5.show_weights(perm, feature_names=...)`

# 4) Introdução Conceitual (Book‑Style)

Após treinar um modelo, surge uma pergunta natural:

> **Quais features o modelo realmente usa para tomar decisões?**

A técnica de **Permutation Importance** responde isso de forma simples:

1. Medimos a performance original do modelo (baseline).  
2. Embaralhamos os valores de **uma única coluna**.  
3. Medimos a nova performance.  
4. A diferença indica o quanto o modelo dependia daquela feature.

Se a performance piorar muito → a feature era importante.  
Se quase não mudar → a feature pouco contribui.

Nesta lesson, aplicaremos esse raciocínio ao dataset sintético
**medical_readmissions.csv**, criado especialmente para este curso.

# 5) Implementação passo a passo

## 5.1. Configuração de Caminhos + Importação de Bibliotecas

In [1]:
import sys
from pathlib import Path

# Caminho absoluto do notebook
notebook_path = Path().resolve()

# Sobe diretórios até encontrar config.py
for parent in notebook_path.parents:
    if (parent / "config.py").exists():
        sys.path.append(str(parent))
        break

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from config import get_data_path, get_output_path

DATA_PATH = get_data_path()
OUTPUT_PATH = get_output_path()

print("DATA_PATH:", DATA_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)

DATA_PATH: ../data/raw/
OUTPUT_PATH: ../outputs/


## 5.2. Carregar o dataset sintético

Dataset criado previamente:
- medical_readmissions.csv

In [3]:
df = pd.read_csv(DATA_PATH + "medical_readmissions.csv")
df.head()

,age,gender,race,admission_type,time_in_hospital,time_in_hospital_scaled,num_procedures,num_medications,lab_tests,bmi_estimate,socks_owned,favorite_color,diabetes,hypertension,readmitted
0,62.450712,Female,Hispanic,Emergency,3,3.0,0,3,45.280360,21.610513,18,Blue,No,Yes,0
1,52.926035,Female,White,Emergency,4,4.0,5,3,32.315645,16.338189,6,Red,Yes,No,0
2,64.715328,Female,Other,Emergency,4,4.0,1,4,40.490947,23.851009,5,Blue,Yes,No,0
3,77.845448,Male,Asian,Urgent,6,6.0,3,4,26.482925,28.906129,4,Green,No,No,0
4,51.487699,Male,Black,Emergency,12,12.0,2,9,46.858905,11.503526,3,Green,Yes,No,0


## 5.3. Separar features e target

Target: `readmitted` (0/1)

In [4]:
y = df["readmitted"]
X = df.drop(columns=["readmitted"])

X.head()

,age,gender,race,admission_type,time_in_hospital,time_in_hospital_scaled,num_procedures,num_medications,lab_tests,bmi_estimate,socks_owned,favorite_color,diabetes,hypertension
0,62.450712,Female,Hispanic,Emergency,3,3.0,0,3,45.280360,21.610513,18,Blue,No,Yes
1,52.926035,Female,White,Emergency,4,4.0,5,3,32.315645,16.338189,6,Red,Yes,No
2,64.715328,Female,Other,Emergency,4,4.0,1,4,40.490947,23.851009,5,Blue,Yes,No
3,77.845448,Male,Asian,Urgent,6,6.0,3,4,26.482925,28.906129,4,Green,No,No
4,51.487699,Male,Black,Emergency,12,12.0,2,9,46.858905,11.503526,3,Green,Yes,No


## 5.4. Pré‑processamento simples

- One‑hot encoding para variáveis categóricas.
- Mantemos tudo simples para focar na explicabilidade.

In [5]:
X = pd.get_dummies(X, drop_first=True)
X.head()

,age,time_in_hospital,time_in_hospital_scaled,num_procedures,num_medications,lab_tests,bmi_estimate,socks_owned,gender_Male,race_Black,race_Hispanic,race_Other,race_White,admission_type_Emergency,admission_type_Urgent,favorite_color_Green,favorite_color_Red,diabetes_Yes,hypertension_Yes
0,62.450712,3,3.0,0,3,45.280360,21.610513,18,False,False,True,False,False,True,False,False,False,False,True
1,52.926035,4,4.0,5,3,32.315645,16.338189,6,False,False,False,False,True,True,False,False,True,True,False
2,64.715328,4,4.0,1,4,40.490947,23.851009,5,False,False,False,True,False,True,False,False,False,True,False
3,77.845448,6,6.0,3,4,26.482925,28.906129,4,True,False,False,False,False,False,True,True,False,False,False
4,51.487699,12,12.0,2,9,46.858905,11.503526,3,True,True,False,False,False,True,False,True,False,True,False


## 5.5. Dividir em treino e validação

In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=0
)

X_train.shape, X_valid.shape

((960, 19), (240, 19))

## 5.6. Treinar o modelo base

Usamos um **RandomForestClassifier** simples.

In [7]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=0
)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [8]:
baseline_preds = model.predict(X_valid)
baseline_acc = accuracy_score(y_valid, baseline_preds)

print(f"Accuracy baseline: {baseline_acc:.4f}")

Accuracy baseline: 0.9750


# 6) Permutation Importance na prática

## 6.1. Implementação manual (didática)

Vamos embaralhar uma única coluna para entender o conceito.

In [9]:
def permutation_importance_single(model, X_valid, y_valid, col_name):
    baseline = accuracy_score(y_valid, model.predict(X_valid))

    X_shuffled = X_valid.copy()
    X_shuffled[col_name] = np.random.permutation(X_shuffled[col_name])

    shuffled_acc = accuracy_score(y_valid, model.predict(X_shuffled))

    return baseline - shuffled_acc, baseline, shuffled_acc

col_example = X_train.columns[0]
imp, base, shuf = permutation_importance_single(model, X_valid, y_valid, col_example)

print(f"Feature: {col_example}")
print(f"Baseline: {base:.4f}")
print(f"Shuffled: {shuf:.4f}")
print(f"Δ Accuracy: {imp:.4f}")

Feature: age
Baseline: 0.9750
Shuffled: 0.9750
Δ Accuracy: 0.0000


## 6.2. Permutation Importance para todas as features

In [10]:
def permutation_importance_all(model, X_valid, y_valid):
    baseline = accuracy_score(y_valid, model.predict(X_valid))
    importances = {}

    for col in X_valid.columns:
        X_shuffled = X_valid.copy()
        X_shuffled[col] = np.random.permutation(X_shuffled[col])

        shuffled_acc = accuracy_score(y_valid, model.predict(X_shuffled))
        importances[col] = baseline - shuffled_acc

    return baseline, importances

baseline, importances = permutation_importance_all(model, X_valid, y_valid)

sorted_importances = sorted(importances.items(), key=lambda x: x[1], reverse=True)

for col, imp in sorted_importances[:10]:
    print(f"{col:30s} Δ Accuracy = {imp:.4f}")

age                            Δ Accuracy = 0.0000
time_in_hospital               Δ Accuracy = 0.0000
time_in_hospital_scaled        Δ Accuracy = 0.0000
num_procedures                 Δ Accuracy = 0.0000
num_medications                Δ Accuracy = 0.0000
lab_tests                      Δ Accuracy = 0.0000
bmi_estimate                   Δ Accuracy = 0.0000
socks_owned                    Δ Accuracy = 0.0000
gender_Male                    Δ Accuracy = 0.0000
race_Black                     Δ Accuracy = 0.0000


## 6.3. Usando `eli5` (como no Kaggle)

A biblioteca `eli5` implementa permutation importance com repetição,
mostrando também a variabilidade.

In [11]:
import eli5
from eli5.sklearn import PermutationImportance

perm = PermutationImportance(
    model,
    random_state=1
).fit(X_valid, y_valid)

eli5.show_weights(perm, feature_names=X_valid.columns.tolist())

/home/moacir/ml/envs/kaggle-ml/lib/python3.12/site-packages/eli5/formatters/html.py:237: RuntimeWarning: invalid value encountered in scalar divide
  rel_weight = (abs(weight) / weight_range) ** 0.7


Weight,Feature
0 ± 0.0000,hypertension_Yes
0 ± 0.0000,diabetes_Yes
0 ± 0.0000,favorite_color_Red
0 ± 0.0000,favorite_color_Green
0 ± 0.0000,admission_type_Urgent
0 ± 0.0000,admission_type_Emergency
0 ± 0.0000,race_White
0 ± 0.0000,race_Other
0 ± 0.0000,race_Hispanic
0 ± 0.0000,race_Black


# 7) Observações pedagógicas do Copilot

- O Kaggle assume que você já domina:
  - treino/validação
  - modelos baseados em árvores
  - one‑hot encoding

- Permutation Importance mede **dependência do modelo**, não causalidade.

- Features irrelevantes (ex.: `socks_owned`) devem aparecer com importância baixa.

- Features redundantes (ex.: `time_in_hospital_scaled`) dividem importância.

- Features correlacionadas (ex.: `bmi_estimate`) podem ter importância diluída.

- Valores negativos podem ocorrer por acaso.

- Esta técnica é ideal para:
  - debugging
  - validação de hipóteses
  - comunicação com stakeholders

- Prepara terreno para:
  - PDP (efeito médio)
  - SHAP (explicações locais)

# 8) Conclusão

Nesta lesson, você:

- entendeu o conceito de feature importance  
- aprendeu a lógica da técnica permutation importance  
- implementou a técnica manualmente  
- aplicou a versão otimizada com `eli5`  
- interpretou importâncias no dataset sintético  

A mensagem principal:

> Permutation Importance revela **quais features o modelo realmente usa**.

Na próxima lesson, veremos:

> **Partial Dependence Plots (PDPs)** — como cada feature afeta as previsões.